# LoRA Fine-Tune — Resume Fit-Summary Generator (Phase 2/3)

Fine-tunes **Llama 3.2 3B Instruct** with **QLoRA** to reproduce the screener's
4-point fit summary. Runs on a free Colab **T4 GPU**.

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU`.
2. Add your HF token as a Colab secret: 🔑 icon (left sidebar) → `New secret` →
   name `HF_TOKEN`, paste your token, enable *Notebook access*.
3. Have `train.jsonl` and `eval.jsonl` ready to upload (from your `data/` folder).

**Strategy:** leave `SMOKE_TEST = True` for the first run — it trains on just 20
examples for 1 epoch (~2 min) to shake out setup/OOM errors. Once that completes
cleanly, set `SMOKE_TEST = False` and run the real fine-tune.

### 1. Check the GPU

In [ ]:
!nvidia-smi

### 2. Install the libraries
(All unpinned → latest mutually-compatible versions that match Colab's current CUDA. Pinning old versions caused bitsandbytes/transformers import clashes.)

In [ ]:
!pip install -q -U transformers trl peft accelerate datasets bitsandbytes
print("done — if Colab shows a 'restart' button, click it, then Runtime -> Restart and continue from cell 3.")

### 3. Authenticate to Hugging Face
Needed to download the gated Llama model.

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
    print("logged in via Colab secret")
except Exception as e:
    print("No HF_TOKEN secret found, falling back to manual prompt:", e)
    login()

### 3b. Mount Google Drive (durable storage)
Saves checkpoints straight to Drive so a disconnect can never lose your model.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs("/content/drive/MyDrive/lora-finetune", exist_ok=True)
print("Drive mounted -> model saves to /content/drive/MyDrive/lora-finetune/")

### 4. Upload your data
Run the cell, then pick **`train.jsonl`** and **`eval.jsonl`** from your computer
(`LoRA FineTuning/data/`). You can select both at once.

In [ ]:
import os
from google.colab import files
if not (os.path.exists("train.jsonl") and os.path.exists("eval.jsonl")):
    uploaded = files.upload()
print("present:", [f for f in ["train.jsonl","eval.jsonl"] if os.path.exists(f)])

### 5. Configuration
All the knobs in one place. This is what you'll tweak in Phase 3.

In [ ]:
SMOKE_TEST   = True          # True = tiny 20-example/1-epoch shakedown run

MODEL_ID     = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR   = "/content/drive/MyDrive/lora-finetune/resume-summary-lora-v2"

# LoRA
LORA_R       = 16            # adapter rank (capacity)
LORA_ALPHA   = 32            # adapter strength (~2x rank is typical)
LORA_DROPOUT = 0.05

# Training (A100 has 40GB headroom -> bigger batch, more epochs)
LEARNING_RATE = 2e-4         # LoRA tolerates a higher LR than full fine-tuning
EPOCHS        = 3            # was 1 on the slow T4; cheap on A100
MAX_SEQ_LEN   = 2048         # resume+JD+summary fits in this window
BATCH_SIZE    = 4            # A100 can handle this; lower to 1-2 if OOM
GRAD_ACCUM    = 4            # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
print("config set. SMOKE_TEST =", SMOKE_TEST)

### 6. Load the model in 4-bit (QLoRA)

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

BF16 = torch.cuda.is_bf16_supported()          # A100=True, T4=False (uses fp16)
compute_dtype = torch.bfloat16 if BF16 else torch.float16
print("bf16 supported:", BF16, "-> compute dtype:", compute_dtype)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=compute_dtype,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print("model loaded:", model.config.model_type)

### 7. Attach the LoRA adapter
Watch the trainable %% — you're training a tiny fraction of the model.

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

### 8. Tokenize with completion-only masking
Loss is computed only on the assistant summary (prompt tokens set to -100).

In [ ]:
/Users/dhruvirathod/Downloads/eval_results_v2.json

### 9. Set up the trainer

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Plain Trainer (not SFTTrainer): we already tokenized + masked labels ourselves.
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True,
                                  label_pad_token_id=-100)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=(1 if SMOKE_TEST else EPOCHS),
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    optim="adamw_torch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=BF16, fp16=not BF16,
    save_strategy="steps", save_steps=20, save_total_limit=2,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
print("trainer ready (completion-only loss)")

### 10. Train 🚀
Smoke ~1 min. Full 3-epoch run ~15-30 min on an A100. Loss should fall well below the old ~2.3.

In [ ]:
trainer.train()

### 11. Save the adapter
Colab is ephemeral — also copy to Drive so you don't lose it.

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
import os
print("saved to (Drive):", OUTPUT_DIR)
print("files:", os.listdir(OUTPUT_DIR))

### 12. Quick sanity check
Generate a summary for one held-out eval example. (After a smoke run the output will be rough — we just want it to *run*.)

In [ ]:
import json
ex = [json.loads(l) for l in open("eval.jsonl")][0]

# return_dict=True + return_tensors -> input_ids with exactly ONE BOS (no double-BOS)
enc = tokenizer.apply_chat_template(
    [ex["messages"][0]], add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
).to(model.device)

# clean inference mode — without these, a freshly-trained model outputs gibberish
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()
out = model.generate(**enc, max_new_tokens=300, do_sample=False,
                     repetition_penalty=1.2, pad_token_id=tokenizer.eos_token_id)
gen = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== MODEL OUTPUT ===\n", gen)
print("\n=== REFERENCE (teacher) ===\n", ex["messages"][1]["content"][:400])

### 13. Full evaluation (all 30) -> Drive
Generates base vs fine-tuned for every held-out example and saves for judging.

In [ ]:
import json, torch, os
model.gradient_checkpointing_disable(); model.config.use_cache=True; model.eval()

def summarize(user_msg, base=False, n=300):
    enc = tokenizer.apply_chat_template([user_msg], add_generation_prompt=True,
            return_tensors="pt", return_dict=True).to(model.device)
    def run():
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=n, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if base:
        with model.disable_adapter(): return run()
    return run()

evalset=[json.loads(l) for l in open("eval.jsonl")]
results=[]
for i,ex in enumerate(evalset):
    um=ex["messages"][0]
    results.append({"fit_label":ex.get("label"),"reference":ex["messages"][1]["content"],
                    "base":summarize(um,base=True),"finetuned":summarize(um,base=False)})
    print(f"[{i+1}/{len(evalset)}] {ex.get('fit_label',ex.get('label'))}")
os.makedirs("/content/drive/MyDrive/lora-finetune", exist_ok=True)
json.dump(results, open("/content/drive/MyDrive/lora-finetune/eval_results_v2.json","w"), indent=2)
print("saved -> Drive/lora-finetune/eval_results_v2.json — download this and send to Claude")

### Next steps
- ✅ Smoke run clean? Set `SMOKE_TEST = False` (cell 5), `Runtime → Restart`, run all → real fine-tune (**Phase 3**).
- Then: full evaluation base-vs-fine-tuned (**Phase 4**), integrate into the screener (**Phase 5**), push to HF Hub (**Phase 6**).
- Hit an error? That's expected — copy it back to Claude and we'll debug together.